# IAU – Fáza 2:  Predspracovanie údajov

## Autori
- Sofiia Berehovykh – 50 %
- Viktoriia Bondarets – 50 %

## 2.1 Realizácia predspracovania dát

### A. Rozdelenie dát na trénovaciu a testovaciu množinu

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    PowerTransformer,
    QuantileTransformer,
    OneHotEncoder
)

if Path("data").is_dir():
    DATA_DIR = Path("data")
elif Path("../data").is_dir():
    DATA_DIR = Path("../data")
else:
    raise FileNotFoundError("Data folder not found")

units = pd.read_csv(DATA_DIR / "units.csv")
readings = pd.read_csv(DATA_DIR / "readings.csv", sep="\t")

readings_clean = readings.drop_duplicates().copy()
units_clean = units.copy()

readings_clean["idUnit"] = pd.to_numeric(
    readings_clean["idUnit"].astype(str).str.extract(r"^U(\d+)$")[0],
    errors="coerce"
)

units_clean["idUnit"] = pd.to_numeric(
    units_clean["idUnit"],
    errors="coerce"
)

readings_clean["tsCycle"] = pd.to_datetime(
    readings_clean["tsCycle"],
    format="mixed",
    errors="coerce"
)

readings_clean["temperatureOil"] = readings_clean[
    "temperatureOil"
].replace(-999, np.nan)

units_clean["dutyClass"] = units_clean["dutyClass"].apply(
    lambda x: x.strip().lower() if isinstance(x, str) else np.nan
)

data = readings_clean.merge(
    units_clean,
    on="idUnit",
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("", "_unit")
)

print("Merge results:")
display(data["_merge"].value_counts())

data = data.drop(columns=["_merge"])

data["cycle_year"] = data["tsCycle"].dt.year
data["cycle_month"] = data["tsCycle"].dt.month
data["cycle_dayofweek"] = data["tsCycle"].dt.dayofweek
data["cycle_hour"] = data["tsCycle"].dt.hour

data = data.drop(
    columns=["idUnit", "tsCycle", "number_serial", "operatorOwner"],
    errors="ignore"
)

data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=["failure"]).copy()
data = data.loc[data["failure"].isin([0, 1])].copy()
data["failure"] = data["failure"].astype(int)

print("Final dataset shape:", data.shape)
print("Duplicate rows:", data.duplicated().sum())
print("Missing values:", data.isna().sum().sum())

display(data.head())

Na základe výsledkov prvej fázy sme upravili dátové súbory `readings.csv` a `units.csv`. Odstránili sme duplicitné záznamy, opravili formáty vybraných atribútov a chýbajúce hodnoty. Následne sme oba súbory spojili pomocou spoločného atribútu `idUnit`.
Súbor `gateways.csv` sme sa rozhodli nepoužiť, pretože väčšina jeho záznamov nemá priradený identifikátor `idUnit`. Spojenie cez `idDepot` by mohlo spôsobiť duplicity a nejednoznačné priradenie údajov. Súbor obsahuje predovšetkým údaje o komunikačných bránach, ktoré neposkytujú priame informácie o stave APU.

In [ ]:
X = data.drop(columns=["failure"])
y = data["failure"]

categorical_candidates = [
    "idDepot",
    "codeLine",
    "code_model",
    "dutyClass"
]

for col in categorical_candidates:
    if col in X.columns:
        X[col] = X[col].apply(
            lambda x: str(x) if pd.notna(x) else np.nan
        )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).sort_index())

print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True).sort_index())

Najprv sme oddelili cieľovú premennú failure od ostatných atribútov, ktoré budeme používať na predikciu. Vybrané kategóriálne atribúty sme previedli na textový formát, aby sme ich neskôr mohli správne zakódovať. Následne sme dáta rozdelili na trénovaciu (80 %) a testovaciu (20 %) množinu. Použili sme stratifikované rozdelenie, aby bol pomer hodnôt failure = 0 a failure = 1 v oboch množinách približne rovnaký. Nakoniec sme rozdelili atribúty na číselné a kategóriálne a skontrolovali počet záznamov a rozdelenie hodnôt premennej failure.

### B. Transformácia dát na vhodný formát pre strojové učenie

In [ ]:
def create_preprocessor(numeric_transformer):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("transformer", numeric_transformer)
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ], remainder="drop")


def transform_datasets(numeric_transformer):
    preprocessor = create_preprocessor(numeric_transformer)

    train_array = preprocessor.fit_transform(X_train)

    feature_names = preprocessor.get_feature_names_out()

    train_result = pd.DataFrame(
        train_array,
        columns=feature_names,
        index=X_train.index
    )

    return preprocessor, train_result

Vytvorili sme dve funkcie na predspracovanie dát pomocou rôznych metód. V číselných atribútoch nahrádzame chýbajúce hodnoty mediánom a následne vykonávame transformáciu. Pri kategóriálnych atribútoch používame najčastejšiu hodnotu a OneHotEncoder na ich prevod na čísla. Pomocou Pipeline a ColumnTransformer tieto kroky automatizujeme.

### C. Transformácia atribútov pomocou rôznych techník

In [ ]:
scaling_methods = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler()
}

scaling_results = {}

for name, scaler in scaling_methods.items():
    preprocessor, train_result = transform_datasets(scaler)

    scaling_results[name] = {
        "preprocessor": preprocessor,
        "X_train": train_result
    }

    print(f"\n{name}")
    print("Train shape:", train_result.shape)
    print("Train missing values:", train_result.isna().sum().sum())
    print(
        "All train features numeric:",
        train_result.select_dtypes(exclude=["number"]).empty
    )

    display(train_result.head())

StandardScaler upravuje hodnoty tak, aby mali priemer približne 0 a štandardnú odchýlku 1, zatiaľ čo MinMaxScaler ich prevádza do intervalu od 0 do 1 na základe minimálnych a maximálnych hodnôt trénovacích dát. Obidve metódy sme použili na rovnaké dáta, aby sme neskôr mohli porovnať ich vplyv na výsledky modelov. Po predspracovaní sme získali 154 numerických atribútov bez chýbajúcich hodnôt, pričom počet záznamov zostal nezmenený. Výsledky oboch metód sme si uložili na ďalšie použitie.

In [ ]:
transformer_methods = {
    "PowerTransformer": PowerTransformer(
        method="yeo-johnson",
        standardize=True
    ),
    "QuantileTransformer": QuantileTransformer(
        n_quantiles=min(1000, len(X_train)),
        output_distribution="normal",
        random_state=42
    )
}

transformer_results = {}

for name, transformer in transformer_methods.items():
    preprocessor, train_result = transform_datasets(transformer)

    transformer_results[name] = {
        "preprocessor": preprocessor,
        "X_train": train_result
    }

    print(f"\n{name}")
    print("Train shape:", train_result.shape)
    print("Train missing values:", train_result.isna().sum().sum())
    print(
        "All train features numeric:",
        train_result.select_dtypes(exclude=["number"]).empty
    )

    display(train_result.head())

PowerTransformer pomocou metódy Yeo-Johnson znižuje asymetriu rozdelenia a následne hodnoty štandardizuje, zatiaľ čo QuantileTransformer transformuje hodnoty na rozdelenie podobné normálnemu. Tieto metódy sme zvolili preto, že v našich senzorových dátach sa nachádzajú aj nerovnomerne rozdelené hodnoty a odľahlé pozorovania. Po oboch transformáciách sme získali 154 numerických atribútov bez chýbajúcich hodnôt, pričom počet záznamov zostal nezmenený. Obe metódy môžu pomôcť znížiť vplyv extrémnych hodnôt, ale zároveň menia pôvodné rozdelenie dát, preto ich vhodnosť posúdime až na základe výsledkov modelov strojového učenia.

In [ ]:
OUTPUT_DIR = Path("faza2_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_results = {
    **scaling_results,
    **transformer_results
}

for name, result in all_results.items():
    train_output = result["X_train"].copy()

    train_output["failure"] = y_train.loc[train_output.index]

    filename = name.lower()

    train_output.to_csv(
        OUTPUT_DIR / f"{filename}_train.csv",
        index=False
    )

    print(f"{name}: train={train_output.shape}")

Výsledky všetkých štyroch transformačných metód sme uložili do samostatných CSV súborov. Ku každému datasetu sme pridali cieľovú premennú failure a zachovali správne priradenie hodnôt k jednotlivým záznamom. Celkovo sme vytvorili štyri súbory, ktoré obsahujú 154 numerických vstupných atribútov a cieľovú premennú.

### D. Zdôvodnenie zvolených postupov predspracovania

Dáta sme rozdelili na trénovaciu (80 %) a testovaciu (20 %) množinu so stratifikáciou, pretože triedy `failure = 0` a `failure = 1` nie sú rovnomerne zastúpené. Chýbajúce numerické hodnoty sme sa rozhodli dopĺňať mediánom, ktorý je menej citlivý na odľahlé hodnoty, a pri kategóriálnych atribútoch sme použili najčastejšiu hodnotu. Kategóriálne atribúty sme previedli na numerické pomocou `OneHotEncoder`. Pri numerických atribútoch sme porovnali štyri metódy: `StandardScaler` a `MinMaxScaler` na úpravu mierky hodnôt, `PowerTransformer` a `QuantileTransformer` na úpravu ich rozdelenia. Keďže v senzorových dátach sa nachádzajú aj extrémne hodnoty, ktoré môžu súvisieť s poruchami zariadení, nechceli sme ich automaticky odstraňovať. Zatiaľ sme nevybrali jednu najlepšiu transformačnú metódu, pretože jej vhodnosť chceme overiť porovnaním výsledkov modelov v ďalšej fáze. Na predspracovanie sme použili `Pipeline` a `ColumnTransformer`, aby boli všetky kroky opakovateľné a aby sa parametre transformácií určovali iba z trénovacích dát.

## 2.2 Výber atribútov pre strojové učenie

In [ ]:
DATA_DIR = Path("faza2_data")
train = pd.read_csv(DATA_DIR / "standardscaler_train.csv")

X = train.drop(columns=["failure"])
y = train["failure"].astype(int)

constant_features = X.columns[X.nunique(dropna=False) <= 1].tolist()
X = X.drop(columns=constant_features)

print("Počet trénovacích pozorovaní:", len(X))
print("Počet atribútov po odstránení konštantných atribútov:", X.shape[1])
print("Odstránené konštantné atribúty:", constant_features)
print("Rozdelenie cieľovej premennej:")
print(y.value_counts())

Na výber najdôležitejších atribútov sme použili trénovaciu množinu predspracovanú pomocou StandardScaler, aby sme pri hodnotení nevyužívali informácie z testovacích dát. Oddelili sme vstupné atribúty od cieľovej premennej failure a následne sme skontrolovali, či niektoré atribúty neobsahujú vo všetkých záznamoch rovnakú hodnotu. Takéto konštantné atribúty sme odstránili, pretože nedokážu rozlišovať medzi záznamami so zlyhaním a bez zlyhania. Po tejto úprave nám zostalo 11 855 trénovacích pozorovaní a 153 atribútov. Zároveň sme skontrolovali rozdelenie cieľovej premennej, keďže počet záznamov s failure = 0 je výrazne vyšší ako počet záznamov s failure = 1.

### A. Identifikácia informatívnych atribútov pomocou troch metód

In [ ]:
f_scores, f_pvalues = f_classif(X, y)

anova_results = pd.DataFrame({
    "feature": X.columns,
    "anova_f": f_scores,
    "anova_pvalue": f_pvalues
}).sort_values("anova_f", ascending=False)

display(anova_results.head(20))

Ako prvú metódu na hodnotenie dôležitosti atribútov sme použili ANOVA F-test, ktorý porovnáva priemerné hodnoty jednotlivých atribútov medzi skupinami failure = 0 a failure = 1. Pre každý atribút sme vypočítali F-skóre a p-hodnotu a výsledky zoradili podľa F-skóre od najvyššieho. Najvyššie hodnoty dosiahli year_commission, serviceHours a dp_filter, čo naznačuje, že tieto atribúty majú výraznejšie rozdiely medzi skupinami.

In [ ]:
mi_scores = mutual_info_classif(
    X,
    y,
    random_state=42,
    n_neighbors=3
)

mi_results = pd.DataFrame({
    "feature": X.columns,
    "mutual_info": mi_scores
}).sort_values("mutual_info", ascending=False)

display(mi_results.head(20))

Ako druhú metódu sme použili Mutual Information, ktorá hodnotí, koľko informácií nám jednotlivé atribúty poskytujú o cieľovej premennej failure. Na rozdiel od ANOVA F-testu dokáže zachytiť aj nelineárne závislosti, preto sme ju zvolili ako doplnkovú metódu. Pre každý atribút sme vypočítali skóre vzájomnej informácie a výsledky zoradili od najvyššej hodnoty. Vyššie skóre naznačuje silnejšiu závislosť medzi atribútom a výskytom zlyhania, zatiaľ čo hodnota blízka nule znamená, že metóda nezistila výraznú závislosť.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=250,
    random_state=42,
    class_weight="balanced_subsample",
    min_samples_leaf=2,
    n_jobs=-1
)

rf.fit(X, y)

rf_results = pd.DataFrame({
    "feature": X.columns,
    "rf_importance": rf.feature_importances_
}).sort_values("rf_importance", ascending=False)

display(rf_results.head(20))

Ako tretiu metódu sme použili Random Forest s 250 rozhodovacími stromami. Keďže triedy failure nie sú vyvážené, použili sme balanced_subsample. Model sme natrénovali na trénovacích dátach a pomocou feature_importances_ sme získali poradie najdôležitejších atribútov.

### B. Zoradenie atribútov podľa dôležitosti

In [ ]:
ranking = pd.DataFrame({"feature": X.columns})

ranking["anova_f"] = f_scores
ranking["mutual_info"] = mi_scores
ranking["rf_importance"] = rf.feature_importances_

ranking["anova_rank"] = ranking["anova_f"].rank(
    ascending=False, method="min"
).astype(int)

ranking["mi_rank"] = ranking["mutual_info"].rank(
    ascending=False, method="min"
).astype(int)

ranking["rf_rank"] = ranking["rf_importance"].rank(
    ascending=False, method="min"
).astype(int)

ranking["average_rank"] = ranking[
    ["anova_rank", "mi_rank", "rf_rank"]
].mean(axis=1)

ranking = ranking.sort_values("average_rank")

display(ranking.head(20).reset_index(drop=True))

In [ ]:
methods = [
    ("anova_f", "ANOVA F-test"),
    ("mutual_info", "Mutual Information"),
    ("rf_importance", "Random Forest")
]

fig, axes = plt.subplots(1, 3, figsize=(21, 9))

for ax, (column, title) in zip(axes, methods):
    top = ranking.sort_values(column, ascending=False).head(15)
    top = top.sort_values(column, ascending=True)

    ax.barh(top["feature"], top[column])
    ax.set_title(title)
    ax.set_xlabel("Importance score")
    ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.show()

### C. Zdôvodnenie výberu atribútov a vyhodnotenie grafov

Na výber dôležitých atribútov sme použili tri metódy: ANOVA F-test, Mutual Information a Random Forest, pretože každá hodnotí atribúty iným spôsobom. Z grafov vidíme, že ANOVA najvyššie hodnotí year_commission a serviceHours, zatiaľ čo pri Mutual Information a Random Forest dominuje dp_filter. Významné umiestnenie vo viacerých metódach dosiahli aj rms_vibration a serviceHours. Výsledky naznačujú, že pri predikcii zlyhania môžu byť dôležité najmä údaje o stave filtrov, vibráciách a dobe prevádzky zariadenia. Keďže skóre jednotlivých metód majú rôzne škály, porovnali sme ich poradia a vypočítali priemerné umiestnenie. V spoločnom rebríčku sa na prvých troch miestach umiestnili dp_filter, serviceHours a rms_vibration. Tieto atribúty preto považujeme za najvhodnejších kandidátov pre ďalšie modelovanie.

## 2.3 Replikovateľnosť predspracovania

### A. Opakovateľné predspracovanie trénovacej a testovacej množiny

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

def create_preprocessing_pipeline(numeric_transformer):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", numeric_transformer)
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return Pipeline([
        ("preprocessing", preprocessor)
    ])

Vytvorili sme funkciu, ktorá spája všetky kroky predspracovania do jedného Pipeline. Numerické atribúty dopĺňame mediánom a následne na ne aplikujeme zvolenú transformačnú metódu. Kategóriálne atribúty dopĺňame najčastejšou hodnotou a pomocou OneHotEncoder ich prevádzame na numerické. Použitím ColumnTransformer zabezpečujeme správne spracovanie oboch typov atribútov.

### B. Vytvorenie a overenie predspracovania pomocou sklearn.pipeline

In [ ]:
transformers = {
    "standardscaler": StandardScaler(),
    "minmaxscaler": MinMaxScaler(),
    "powertransformer": PowerTransformer(
        method="yeo-johnson",
        standardize=True
    ),
    "quantiletransformer": QuantileTransformer(
        n_quantiles=min(1000, len(X_train)),
        output_distribution="normal",
        random_state=42
    )
}

OUTPUT_DIR = Path("faza2_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

preprocessing_pipelines = {}
preprocessed_data = {}

for name, transformer in transformers.items():
    pipeline = create_preprocessing_pipeline(transformer)

    X_train_transformed = pipeline.fit_transform(X_train)
    X_test_transformed = pipeline.transform(X_test)

    feature_names = pipeline.get_feature_names_out()

    train_output = pd.DataFrame(
        X_train_transformed,
        columns=feature_names,
        index=X_train.index
    )
    train_output["failure"] = y_train.to_numpy()

    test_output = pd.DataFrame(
        X_test_transformed,
        columns=feature_names,
        index=X_test.index
    )
    test_output["failure"] = y_test.to_numpy()

    train_output.to_csv(
        OUTPUT_DIR / f"{name}_train.csv",
        index=False
    )
    test_output.to_csv(
        OUTPUT_DIR / f"{name}_test.csv",
        index=False
    )

    preprocessing_pipelines[name] = pipeline
    preprocessed_data[name] = {
        "train": train_output,
        "test": test_output
    }

    print(
        f"{name}: train={train_output.shape}, "
        f"test={test_output.shape}"
    )

Vytvorené Pipeline sme otestovali so všetkými štyrmi transformačnými metódami. Pre každú metódu sme použili fit_transform() na trénovacej množine a transform() na testovacej množine, aby sa parametre predspracovania určovali iba z trénovacích dát. Následne sme výsledky previedli na tabuľky, pridali cieľovú premennú failure a uložili ich do samostatných CSV súborov. Zároveň sme si uložili jednotlivé pipeline na ďalšie použitie v rámci notebooku. Takto sme overili, že rovnaký postup predspracovania môžeme opakovane aplikovať na trénovacie aj testovacie dáta bez potreby meniť kód.

In [ ]:
validation_results = []

for name, datasets in preprocessed_data.items():
    train_output = datasets["train"]
    test_output = datasets["test"]

    train_features = train_output.drop(columns=["failure"])
    test_features = test_output.drop(columns=["failure"])

    validation_results.append({
        "transformer": name,
        "train_rows": len(train_output),
        "test_rows": len(test_output),
        "train_features": train_features.shape[1],
        "test_features": test_features.shape[1],
        "same_features": list(train_features.columns) == list(test_features.columns),
        "train_missing": int(train_features.isna().sum().sum()),
        "test_missing": int(test_features.isna().sum().sum()),
        "target_preserved": (
            train_output["failure"].notna().all()
            and test_output["failure"].notna().all()
        )
    })

validation_results = pd.DataFrame(validation_results)
display(validation_results)

Na záver sme skontrolovali výsledky predspracovania pre všetky štyri transformačné metódy. Overovali sme počet záznamov a atribútov v trénovacej a testovacej množine, či majú obe množiny rovnaké atribúty v rovnakom poradí, či sa v nich nenachádzajú chýbajúce hodnoty a či sa zachovala cieľová premenná failure. 